## Datalab Semester 2, Sprint 3

In [6]:
import os
import sqlite3
import pandas as pd

# 1. Zoek de map op waar dit notebook-bestand staat
map_van_notebook = os.path.dirname(os.path.abspath('__file__'))

# 2. Maak het volledige pad naar de database
db_pad = os.path.join(map_van_notebook, 'database.sqlite')
conn = sqlite3.connect(db_pad)

Bepaal met behulp van SQL in deze sprint het volgende:

1A Toon het aantal wedstrijden dat jouw team heeft gespeeld per seizoen.

In [7]:
query = "SELECT name FROM sqlite_master WHERE type='table';"
tabellen = pd.read_sql_query(query, conn)

print(tabellen)
query_zoek_team = "SELECT team_long_name FROM Team WHERE team_long_name LIKE '%Barcelona%';"
barca_naam = pd.read_sql_query(query_zoek_team, conn)

print(barca_naam)

Empty DataFrame
Columns: [name]
Index: []


DatabaseError: Execution failed on sql 'SELECT team_long_name FROM Team WHERE team_long_name LIKE '%Barcelona%';': no such table: Team

In [3]:
query = """
SELECT 
    season, 
    COUNT(*) AS aantal_wedstrijden
FROM 
    Match
WHERE 
    home_team_api_id = (SELECT team_api_id FROM Team WHERE team_long_name = 'FC Barcelona')
    OR 
    away_team_api_id = (SELECT team_api_id FROM Team WHERE team_long_name = 'FC Barcelona')
GROUP BY 
    season;
"""

df_wedstrijden = pd.read_sql_query(query, conn)
df_wedstrijden

,season,aantal_wedstrijden
0,2008/2009,38
1,2009/2010,38
2,2010/2011,38
3,2011/2012,38
4,2012/2013,38
5,2013/2014,38
6,2014/2015,38
7,2015/2016,38


2A maak een nieuwe dataframe

Dataframe met team eigenschappen

In [5]:
query_2 = "SELECT * FROM player_Attributes"

df_player_Attribute = pd.read_sql_query(query_2, conn)
df_player_Attribute.head()

DatabaseError: Execution failed on sql 'SELECT * FROM player_Attributes': no such table: player_Attributes

In [ ]:
def genereer_competitie_ranglijst(connection, seizoen):
    """
    Haalt wedstrijddata op uit de database en genereert een gesorteerde ranglijst.
    
    Args:
        connection (sqlite3.Connection): De actieve database verbinding.
        seizoen (str): Het gewenste seizoen (bijv. '2015/2016').
        
    Returns:
        pd.DataFrame: Een dataframe met de teamnamen en hun totale punten, gesorteerd van hoog naar laag.
    """
    # 1. Data ophalen
    query = f"SELECT home_team_api_id, away_team_api_id, home_team_goal, away_team_goal FROM Match WHERE season = '{seizoen}'"
    df_matches = pd.read_sql_query(query, connection)
    
    # 2. Punten berekenen met de hulpfunctie
    df_matches[['home_points', 'away_points']] = df_matches.apply(bepaal_match_punten, axis=1)
    
    # 3. Groeperen en totalen berekenen
    home_stats = df_matches.groupby('home_team_api_id')['home_points'].sum().reset_index()
    away_stats = df_matches.groupby('away_team_api_id')['away_points'].sum().reset_index()
    
    home_stats.columns = ['team_api_id', 'points']
    away_stats.columns = ['team_api_id', 'points']
    
    ranglijst = pd.concat([home_stats, away_stats]).groupby('team_api_id').sum().reset_index()
    
    # 4. Teamnamen toevoegen
    df_teams_names = pd.read_sql_query("SELECT team_api_id, team_long_name FROM Team", connection)
    ranglijst = ranglijst.merge(df_teams_names, on='team_api_id')
    
    return ranglijst.sort_values(by='points', ascending=False).reset_index(drop=True)

display()